# 🤟 Train model VSL (Ngôn ngữ ký hiệu Việt Nam)

Chạy **miễn phí trên Google Colab**. Bấm ▶️ từng ô **từ trên xuống**.

1. Cài thư viện
2. Tải dataset
3. Giải nén + tự dò nhãn (xem tóm tắt)
4. Trích landmark (lâu nhất)
5. Train model
6. Xuất model + tải về máy

> Mẹo: **Runtime → Change runtime type → T4 GPU** cho nhanh.


## 1️⃣ Cài thư viện (~2-3 phút)


In [ ]:
# Không pin version cứng -> pip tự chọn bản hợp với Python của Colab.
# tf-keras = Keras 2 (bộ chuyển TF.js chạy ổn định với bản này).
!pip -q install mediapipe tensorflow tf-keras tensorflowjs \
  huggingface_hub opencv-python-headless tqdm scikit-learn pandas

# Kiểm tra MediaPipe Tasks API (API mới, KHÔNG dùng mp.solutions cũ)
from mediapipe.tasks.python import vision
print('\n✅ Cài xong, MediaPipe Tasks OK. Chạy tiếp các ô phía dưới.')


## 2️⃣ Tải dataset (~1.14GB)


In [ ]:
import os, glob
from huggingface_hub import snapshot_download

DATASET_REPO = 'star092304/ViSignLanguage-Video'  # đổi nếu repo khác
DATA_DIR = '/content/vsl_data'
path = snapshot_download(repo_id=DATASET_REPO, repo_type='dataset', local_dir=DATA_DIR)
print('Đã tải về:', path)
print('File:', os.listdir(DATA_DIR))


## 3️⃣ Giải nén + tự dò nhãn

Ô này giải nén `dataset.zip`, đọc `dataset_metadata.csv`, tự tìm cột tên-file và cột nhãn.
**Xem phần tóm tắt cuối ô**: số nhãn nên ~100, mỗi nhãn vài chục video.


In [ ]:
import os, glob, zipfile, collections
import pandas as pd

DATA_DIR='/content/vsl_data'; VID_DIR='/content/vsl_videos'
VIDEO_EXTS=('.mp4','.mov','.avi','.mkv','.webm')
def find_videos(root): return [p for p in glob.glob(os.path.join(root,'**','*'),recursive=True) if p.lower().endswith(VIDEO_EXTS)]
def stem(p): return os.path.splitext(os.path.basename(p))[0]
def base(p): return os.path.basename(p)

# --- giải nén dataset.zip (1 lần) ---
if not find_videos(VID_DIR):
    os.makedirs(VID_DIR,exist_ok=True)
    print('Đang giải nén dataset.zip ...')
    with zipfile.ZipFile(os.path.join(DATA_DIR,'dataset.zip')) as z: z.extractall(VID_DIR)
videos=find_videos(VID_DIR)
print('Tổng video:',len(videos))
assert videos, 'Không thấy video sau khi giải nén — kiểm tra lại dataset.zip'

# --- đọc metadata, dò cột tên-file & cột nhãn ---
meta=pd.read_csv(os.path.join(DATA_DIR,'dataset_metadata.csv'))
meta.columns=[str(c).strip() for c in meta.columns]
print('Cột metadata:',list(meta.columns))
stems={stem(p):p for p in videos}; bases={base(p):p for p in videos}

fname_col=None
for c in meta.columns:
    vals=meta[c].astype(str)
    hit=sum((stem(v) in stems) or (v in bases) or (base(v) in bases) for v in vals)
    if hit>len(videos)*0.4: fname_col=c; break

HINTS=['label','gloss','word','class','text','meaning','vietnamese','sign','name','category','glosa','nhan','tu','y_nghia']
label_col=None
for c in meta.columns:
    if c==fname_col: continue
    if any(h in c.lower() for h in HINTS) and 2<=meta[c].nunique()<=len(videos): label_col=c; break
if label_col is None:
    best=None
    for c in meta.columns:
        if c==fname_col: continue
        nun=meta[c].nunique()
        if 2<=nun<=max(2,len(videos)//3):
            if best is None or abs(nun-100)<abs(meta[best].nunique()-100): best=c
    label_col=best

# --- xây danh sách (video, nhãn) ---
samples=[]; used_meta=False
if fname_col and label_col:
    m={}
    for _,r in meta.iterrows():
        lab=str(r[label_col])
        for k in (stem(str(r[fname_col])), base(str(r[fname_col])), str(r[fname_col])): m[k]=lab
    for p in videos:
        lab=m.get(stem(p)) or m.get(base(p))
        if lab is not None: samples.append((p,lab))
    used_meta=len(samples)>len(videos)*0.5

if not used_meta:  # fallback: nhãn theo thư mục cha (bỏ qua tên split)
    SPLIT={'train','val','valid','test','dataset','videos','data','vsl_videos'}
    samples=[]
    for p in videos:
        parts=os.path.normpath(p).split(os.sep); lab=None
        for d in reversed(parts[:-1]):
            if d.lower() not in SPLIT: lab=d; break
        samples.append((p,lab or 'unknown'))

labs=collections.Counter(l for _,l in samples)
print('\n================ TÓM TẮT ================')
print('Nguồn nhãn:', f'metadata (file={fname_col!r}, nhãn={label_col!r})' if used_meta else 'thư mục cha')
print('Số mẫu có nhãn:',len(samples),'| Số nhãn:',len(labs))
print('Phân bố 20 nhãn đầu:')
for k,v in list(labs.items())[:20]: print(f'   {k}: {v}')
print('Ví dụ mapping:')
for p,l in samples[:8]: print(f'   {os.path.relpath(p,VID_DIR)} -> {l}')


## 4️⃣ Trích landmark (lâu nhất, ~30-90 phút)

Dùng **MediaPipe Tasks API** (cùng model `hand_landmarker.task` mà web app dùng).
Lưu checkpoint mỗi 300 video — nếu Colab rớt giữa chừng, chạy lại ô này sẽ **tự resume**, không mất công.


In [ ]:
import numpy as np, cv2, json, os, urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision
from tqdm.auto import tqdm

# ===== Hằng số (PHẢI khớp app: vsl-web/lib/constants.ts) =====
NUM_HANDS=2; NUM_LANDMARKS=21; COORDS=3
FEATURES_PER_HAND=NUM_LANDMARKS*COORDS; FEATURES_PER_FRAME=NUM_HANDS*FEATURES_PER_HAND
SEQ_LEN=30; WRIST=0; MIDDLE_MCP=9; SAMPLE_FRAMES=36

# ----- Tải model hand_landmarker (giống hệt model web app dùng) -----
TASK='/content/hand_landmarker.task'
if not os.path.exists(TASK):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task', TASK)
landmarker = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=TASK),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=NUM_HANDS, min_hand_detection_confidence=0.5))

def normalize_hand(points):
    pts=points.astype(np.float32).copy(); pts-=pts[WRIST].copy()
    s=np.linalg.norm(pts[MIDDLE_MCP]) or 1.0; pts/=s; return pts.reshape(-1)
def build_frame(hands):
    f=np.zeros(FEATURES_PER_FRAME,dtype=np.float32); slot={'Left':0,'Right':1}
    for h in hands:
        i=slot.get(h['label'])
        if i is None: continue
        f[i*FEATURES_PER_HAND:(i+1)*FEATURES_PER_HAND]=normalize_hand(h['points'])
    return f
def resample(frames,n):
    T=len(frames)
    if T==0: return np.zeros((n,FEATURES_PER_FRAME),dtype=np.float32)
    if T==n: return np.array(frames,dtype=np.float32)
    return np.array(frames,dtype=np.float32)[np.linspace(0,T-1,n).round().astype(int)]
def sample_idx(total,k):
    if total<=0: return None
    if total<=k: return set(range(total))
    return set(np.linspace(0,total-1,k).round().astype(int))
def extract(path):
    cap=cv2.VideoCapture(path); total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    want=sample_idx(total,SAMPLE_FRAMES); frames=[]; i=0
    while True:
        ok,fr=cap.read()
        if not ok: break
        if want is None or i in want:
            img=mp.Image(image_format=mp.ImageFormat.SRGB,
                         data=np.ascontiguousarray(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB)))
            res=landmarker.detect(img); hl=[]
            for lms,hd in zip(res.hand_landmarks,res.handedness):
                hl.append({'label':hd[0].category_name,
                           'points':np.array([[p.x,p.y,p.z] for p in lms],dtype=np.float32)})
            if hl: frames.append(build_frame(hl))
        i+=1
    cap.release(); return frames

samples=sorted(samples)  # thứ tự cố định để resume được
labels_sorted=sorted({l for _,l in samples})
label_to_idx={l:i for i,l in enumerate(labels_sorted)}
print(len(samples),'video,',len(labels_sorted),'nhãn')

# ----- Resume nếu có checkpoint (lưu mỗi 300 video) -----
start=0; X=[]; y=[]
if os.path.exists('/content/ckpt_n.txt'):
    start=int(open('/content/ckpt_n.txt').read())
    X=list(np.load('/content/X.npy')); y=list(np.load('/content/y.npy'))
    print(f'↩️  Resume từ video {start} (đã có {len(X)} mẫu)')

for n,(p,lab) in enumerate(tqdm(samples,desc='Trích landmark',initial=start,total=len(samples))):
    if n<start: continue
    fr=extract(p)
    if fr:
        X.append(resample(fr,SEQ_LEN)); y.append(label_to_idx[lab])
    if (n+1)%300==0:
        np.save('/content/X.npy',np.array(X,dtype=np.float32))
        np.save('/content/y.npy',np.array(y,dtype=np.int64))
        open('/content/ckpt_n.txt','w').write(str(n+1))

X=np.array(X,dtype=np.float32); y=np.array(y,dtype=np.int64)
np.save('/content/X.npy',X); np.save('/content/y.npy',y)
open('/content/ckpt_n.txt','w').write(str(len(samples)))
json.dump(labels_sorted,open('/content/labels.json','w'),ensure_ascii=False,indent=2)
print(f'\n✅ Xong. X={X.shape}, y={y.shape}, nhãn={len(labels_sorted)}')



## 5️⃣ Train model (LSTM nhẹ, vài phút)


In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS']='1'  # Keras 2 cho khớp bộ chuyển TF.js
import numpy as np, json, tensorflow as tf
from sklearn.model_selection import train_test_split

X=np.load('/content/X.npy'); y=np.load('/content/y.npy')
labels=json.load(open('/content/labels.json')); num_classes=len(labels)
print('X',X.shape,'classes',num_classes)
SEQ_LEN=X.shape[1]; FEATURES_PER_FRAME=X.shape[2]

Xtr,Xval,ytr,yval=train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
model=tf.keras.Sequential([
    tf.keras.layers.Input((SEQ_LEN,FEATURES_PER_FRAME)),
    tf.keras.layers.Masking(0.0),
    tf.keras.layers.LSTM(128,return_sequences=True), tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(64), tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(num_classes,activation='softmax'),
])
model.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy'])
cbs=[tf.keras.callbacks.EarlyStopping(patience=12,restore_best_weights=True,monitor='val_accuracy'),
     tf.keras.callbacks.ReduceLROnPlateau(patience=5,factor=0.5)]
model.fit(Xtr,ytr,validation_data=(Xval,yval),epochs=120,batch_size=32,callbacks=cbs)
print('\nVal accuracy:',round(float(model.evaluate(Xval,yval,verbose=0)[1]),3))
model.save('/content/vsl.h5')


## 6️⃣ Xuất model TF.js + tải về máy

Chạy xong tự tải **`vsl-model.zip`**. Gửi file đó cho người hỗ trợ để ghép vào app.


In [ ]:
import os, numpy as np
os.environ['TF_USE_LEGACY_KERAS']='1'

# Vá numpy: tensorflowjs cũ còn dùng np.object/np.bool đã bị numpy mới xoá
for _n,_t in {'object':object,'bool':bool,'int':int,'float':float,'str':str,'complex':complex}.items():
    if not hasattr(np,_n): setattr(np,_n,_t)

import tensorflowjs as tfjs, tensorflow as tf, shutil
m=tf.keras.models.load_model('/content/vsl.h5')
os.makedirs('/content/out/models/vsl',exist_ok=True)
tfjs.converters.save_keras_model(m,'/content/out/models/vsl')
shutil.copy('/content/labels.json','/content/out/labels.json')
shutil.make_archive('/content/vsl-model','zip','/content/out')
print('✅ Đã tạo /content/vsl-model.zip')
from google.colab import files
files.download('/content/vsl-model.zip')

